## 1. Importing modules and basic settings

In [16]:


from datetime import datetime
from pathlib import Path

STUDENT_ID = "850008182"

MAIN_FILE = Path(f"LHMS_{STUDENT_ID}.txt")
GST_RATE = 0.15


# rooms will store room information by room number.
# allocations will store current bookings by room number.
rooms = {}
allocations = {}


print("Room allocation file will be:", MAIN_FILE)

Room allocation file will be: LHMS_850008182.txt


## 2. To Display and validation helper functions
I am creating this small helper functions to avoid repeated codes

In [17]:
def print_line(symbol="-", width=70):

    print(symbol * width)


def print_heading(title):
    """ for printing a simple heading for console sections."""
    print_line("=")
    print(title.center(70))
    print_line("=")


def clean_room_number(value):
    """
    To convert a room number into a clean string.

    Raises:
        ValueError: if the value is blank or not numeric.
    """
    room_no = str(value).strip()

    if room_no == "":
        raise ValueError("Room number cannot be blank.")

    if not room_no.isdigit():
        raise ValueError("Room number must contain digits only.")

    return room_no


def clean_positive_float(value, field_name):
    """
    To Convert a value into a positive float.

    Used for nightly room rate.
    """
    number = float(value)

    if number <= 0:
        raise ValueError(f"{field_name} must be greater than zero.")

    return number


def clean_positive_int(value, field_name):
    """
    To Convert a value into a positive integer.

    Used for number of nights.
    """
    number = int(value)

    if number <= 0:
        raise ValueError(f"{field_name} must be greater than zero.")

    return number

## 3. Safe keyboard input helpers

these are only for the interactive menu.  
The core functions below accept parameters, which makes them easier to test in notebook cells first.

In [3]:
def ask_text(prompt):
    """Keeps asking until the user enters non-blank text."""
    while True:
        text = input(prompt).strip()
        if text:
            return text
        print("Input cannot be blank.")


def ask_room_number(prompt):
    """Asks for a room number and validates it."""
    while True:
        try:
            return clean_room_number(input(prompt))
        except ValueError as error:
            print("Invalid room number:", error)


def ask_float(prompt, field_name):
    """Asks for a positive float value."""
    while True:
        try:
            return clean_positive_float(input(prompt), field_name)
        except ValueError as error:
            print("Invalid number:", error)


def ask_int(prompt, field_name):
    """Asks for a positive integer value."""
    while True:
        try:
            return clean_positive_int(input(prompt), field_name)
        except ValueError as error:
            print("Invalid number:", error)

## 4. Menu Option 1 - Add Room



In [18]:
def add_room(room_no, room_type, nightly_rate, features):

    try:
        room_key = clean_room_number(room_no)
        rate = clean_positive_float(nightly_rate, "Nightly rate")
        room_type = str(room_type).strip().title()
        features = str(features).strip()

        if room_type == "":
            raise ValueError("Room type cannot be blank.")

        if room_key in rooms:
            return f"Room {room_key} already exists."

        rooms[room_key] = {
            "room_no": room_key,
            "room_type": room_type,
            "nightly_rate": rate,
            "features": features,
            "status": "Available"
        }

        return f"Room {room_key} added successfully."

    except (ValueError, TypeError) as error:
        return f"Room was not added. Reason: {error}"


def menu_add_room():
    """Keyboard-input version of menu option 1."""
    print_heading("ADD ROOM")
    room_no = ask_room_number("Enter room number: ")
    room_type = ask_text("Enter room type: ")
    nightly_rate = ask_float("Enter nightly rate: ", "Nightly rate")
    features = ask_text("Enter room features: ")
    print(add_room(room_no, room_type, nightly_rate, features))

### Quick test for Menu Option 1

In [19]:
print(add_room("501", "super-deluxe", 320, "etc etc"))
print(add_room("502", "standard", 245, "hill view"))
print(add_room("501", "super-deluxe", 320, "Duplicate test"))
rooms

Room 501 added successfully.
Room 502 added successfully.
Room 501 already exists.


{'501': {'room_no': '501',
  'room_type': 'Super-Deluxe',
  'nightly_rate': 320.0,
  'features': 'etc etc',
  'status': 'Available'},
 '502': {'room_no': '502',
  'room_type': 'Standard',
  'nightly_rate': 245.0,
  'features': 'hill view',
  'status': 'Available'}}

## 5. Menu Option 2 - Delete Room


- Validates the room number.
- Checks if room exists.
- Do not delete if the room is currently allocated.
- Delete only available rooms.

In [20]:
def delete_room(room_no):
    """Delete an existing room if it is not currently allocated."""
    try:
        room_key = clean_room_number(room_no)

        if room_key not in rooms:
            return f"Room {room_key} does not exist."

        if room_key in allocations or rooms[room_key]["status"] == "Allocated":
            return f"Room {room_key} is allocated and cannot be deleted."

        del rooms[room_key]
        return f"Room {room_key} deleted successfully."

    except (ValueError, TypeError) as error:
        return f"Room was not deleted. Reason: {error}"


def menu_delete_room():
    """Keyboard-input version of menu option 2."""
    print_heading("DELETE ROOM")
    room_no = ask_room_number("Enter room number to delete: ")
    print(delete_room(room_no))

In [21]:
print(delete_room("999"))  # room does not exist
print(delete_room("502"))  # deletes the available room
rooms

Room 999 does not exist.
Room 502 deleted successfully.


{'501': {'room_no': '501',
  'room_type': 'Super-Deluxe',
  'nightly_rate': 320.0,
  'features': 'etc etc',
  'status': 'Available'}}

## 6. Menu Option 3 - Display Room Details

This function can be written without keyboard input because it only displays current room information.

In [22]:
def display_rooms():

    if not rooms:
        print("No room records are currently available.")
        return

    print_heading("ROOM DETAILS")

    for room_no, room in rooms.items():
        print(f"Room Number : {room_no}")
        print(f"Room Type   : {room['room_type']}")
        print(f"Nightly Rate: ${room['nightly_rate']:.2f}")
        print(f"Features    : {room['features']}")
        print(f"Status      : {room['status']}")
        print_line()


def menu_display_rooms():
    """Keyboard-menu wrapper for option 3."""
    display_rooms()

In [23]:
display_rooms()

                             ROOM DETAILS                             
Room Number : 501
Room Type   : Super-Deluxe
Nightly Rate: $320.00
Features    : etc etc
Status      : Available
----------------------------------------------------------------------


## 7. Menu Option 4 - Allocating Room


- Validating the room number.
- Checking if the room exists.
- Checking status is Available or not.
- Collecting customer name, phone/contact, and number of nights.
- Saving booking data in `allocations`.
- Changed room status to Allocated.

In [24]:
def allocate_room(room_no, customer_name, contact_number, nights):
    """Allocate an available room to a guest."""
    try:
        room_key = clean_room_number(room_no)
        guest = str(customer_name).strip()
        contact = str(contact_number).strip()
        stay_length = clean_positive_int(nights, "Number of nights")

        if room_key not in rooms:
            return f"Room {room_key} does not exist."

        if room_key in allocations or rooms[room_key]["status"] == "Allocated":
            return f"Room {room_key} is already allocated."

        if not guest:
            return "Customer name cannot be blank."

        if not contact:
            return "Contact number cannot be blank."

        allocations[room_key] = {
            "customer_name": guest,
            "contact_number": contact,
            "nights": stay_length
        }
        rooms[room_key]["status"] = "Allocated"

        return f"Room {room_key} allocated successfully to {guest}."

    except (ValueError, TypeError) as error:
        return f"Room was not allocated. Reason: {error}"


def menu_allocate_room():
    """Keyboard-input version of menu option 4."""
    print_heading("ALLOCATE ROOM")
    room_no = ask_room_number("Enter room number to allocate: ")
    customer_name = ask_text("Enter customer name: ")
    contact_number = ask_text("Enter contact number: ")
    nights = ask_int("Enter number of nights: ", "Number of nights")
    print(allocate_room(room_no, customer_name, contact_number, nights))

In [25]:
print(allocate_room("501", "Alex Morgan", "0211234567", 3))
print(allocate_room("501", "Another Guest", "0217654321", 2))
display_rooms()

Room 501 allocated successfully to Alex Morgan.
Room 501 is already allocated.
                             ROOM DETAILS                             
Room Number : 501
Room Type   : Super-Deluxe
Nightly Rate: $320.00
Features    : etc etc
Status      : Allocated
----------------------------------------------------------------------


## 8. Menu Option 5 - Display Room Allocation Details


- If no allocations exist, print a clear message.
- Otherwise print every current booking.

In [26]:
def display_allocations():
    """Display all current room allocations."""
    if not allocations:
        print("No room allocation details are currently available.")
        return

    print_heading("ROOM ALLOCATION DETAILS")

    for room_no, booking in allocations.items():
        room = rooms.get(room_no, {})
        print(f"Room Number : {room_no}")
        print(f"Customer    : {booking['customer_name']}")
        print(f"Contact     : {booking['contact_number']}")
        print(f"Nights      : {booking['nights']}")
        print(f"Room Type   : {room.get('room_type', 'N/A')}")
        print(f"Nightly Rate: ${room.get('nightly_rate', 0):.2f}")
        print_line()


def menu_display_allocations():
    """Keyboard-menu wrapper for option 5."""
    display_allocations()

In [27]:
display_allocations()

                       ROOM ALLOCATION DETAILS                        
Room Number : 501
Customer    : Alex Morgan
Contact     : 0211234567
Nights      : 3
Room Type   : Super-Deluxe
Nightly Rate: $320.00
----------------------------------------------------------------------


## 9. Menu Option 6 - Billing and De-Allocation


- Find the booking for the selected room.
- Calculate subtotal = nightly_rate × nights.
- Calculate GST and total.
- Remove the allocation.
- Change room status back to Available.

In [29]:
def bill_and_deallocate(room_no):
    """Calculatiion of the guest bill, remove the allocation, and release the room."""
    try:
        room_key = clean_room_number(room_no)

        if room_key not in rooms:
            return f"Room {room_key} does not exist."

        if room_key not in allocations:
            return f"Room {room_key} has no current allocation."

        room = rooms[room_key]
        booking = allocations[room_key]

        rate = clean_positive_float(room["nightly_rate"], "Nightly rate")
        nights = clean_positive_int(booking["nights"], "Number of nights")

        subtotal = rate * nights
        gst = subtotal * GST_RATE
        total = subtotal + gst

        bill = (
            f"Room Number : {room_key}\n"
            f"Customer    : {booking['customer_name']}\n"
            f"Nights      : {nights}\n"
            f"Subtotal    : ${subtotal:.2f}\n"
            f"GST (15%)   : ${gst:.2f}\n"
            f"Total       : ${total:.2f}\n"
            "Billing completed. Room is now Available."
        )

        del allocations[room_key]
        room["status"] = "Available"

        return bill

    except (KeyError, ValueError, TypeError) as error:
        return f"Billing could not be completed. Reason: {error}"


def menu_bill_and_deallocate():
    """Keyboard-input version of menu option 6."""
    print_heading("BILLING AND DE-ALLOCATION")
    room_no = ask_room_number("Enter room number for billing: ")
    print(bill_and_deallocate(room_no))

In [30]:
print(bill_and_deallocate("501"))
display_rooms()
display_allocations()

Room Number : 501
Customer    : Alex Morgan
Nights      : 3
Subtotal    : $960.00
GST (15%)   : $144.00
Total       : $1104.00
Billing completed. Room is now Available.
                             ROOM DETAILS                             
Room Number : 501
Room Type   : Super-Deluxe
Nightly Rate: $320.00
Features    : etc etc
Status      : Available
----------------------------------------------------------------------
No room allocation details are currently available.


In [31]:
print(allocate_room("501", "Alex Morgan", "0211234567", 2))

Room 501 allocated successfully to Alex Morgan.


## 10. Report builder for file I/O

This helper converts current allocation data into text before saving it to a file.

In [12]:
def build_allocation_report():

    lines = []
    lines.append("LANGHAM HOTEL MANAGEMENT SYSTEM")
    lines.append("ROOM ALLOCATION REPORT")
    lines.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    lines.append("=" * 10)

    if not allocations:
        lines.append("No current room allocations.")
    else:
        for room_no, booking in allocations.items():
            room = rooms.get(room_no, {})
            lines.append(f"Room Number : {room_no}")
            lines.append(f"Customer    : {booking.get('customer_name', 'N/A')}")
            lines.append(f"Contact     : {booking.get('contact_number', 'N/A')}")
            lines.append(f"Nights      : {booking.get('nights', 'N/A')}")
            lines.append(f"Rate        : ${room.get('nightly_rate', 0):.2f}")
            lines.append("=" * 60)

    return "\n".join(lines)

## 11. Menu Options 7, 8, and 9 - File I/O



In [36]:
def save_allocations_to_file():
    """Saving the current allocation report to the student file."""
    try:
        report = build_allocation_report()

        with MAIN_FILE.open("w", encoding="utf-8") as file:
            file.write(report)

        print(f"Allocation report saved to {MAIN_FILE}.")

    except OSError as error:
        print(f"File could not be written: {error}")


def display_allocations_from_file():
    """Read and display the saved allocation report."""
    try:
        with MAIN_FILE.open("r", encoding="utf-8") as file:
            content = file.read()

        if content.strip():
            print(content)
        else:
            print("The allocation file is empty.")

    except FileNotFoundError:
        print("Allocation file was not found. Save data first.")
    except OSError as error:
        print(f"File could not be read: {error}")


def backup_and_clear_allocation_file():
    """Back up the allocation file, then clear the original."""
    try:
        if not MAIN_FILE.exists():
            print("Main allocation file does not exist. Save data first.")
            return

        with MAIN_FILE.open("r", encoding="utf-8") as file:
            content = file.read()

        if not content.strip():
            print("Allocation file is already empty. No backup was created.")
            return

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        backup_file = Path(f"LHMS_{STUDENT_ID}_Backup_{timestamp}.txt")

        with backup_file.open("a", encoding="utf-8") as file:
            file.write(content)

        with MAIN_FILE.open("w", encoding="utf-8") as file:
            file.write("")

        print(f"Backup created: {backup_file}")
        print(f"Original file cleared: {MAIN_FILE}")

    except FileNotFoundError as error:
        print(f"File was not found: {error}")
    except OSError as error:
        print(f"File operation failed: {error}")

In [33]:
save_allocations_to_file()

Allocation report saved to LHMS_850008182.txt.


In [35]:
display_allocations_from_file()

LANGHAM HOTEL MANAGEMENT SYSTEM
ROOM ALLOCATION REPORT
Generated: 2026-10-30 18:07:04
Room Number : 501
Customer    : Alex Morgan
Contact     : 0211234567
Nights      : 2
Rate        : $320.00


In [37]:
backup_and_clear_allocation_file()
display_allocations_from_file()

Backup created: LHMS_850008182_Backup_20261030_181125.txt
Original file cleared: LHMS_850008182.txt
The allocation file is empty.


## 12. Main menu loop


In [14]:
def show_menu():
    """Display the main application menu."""
    print()
    print_line("*")
    print("LANGHAM HOTEL MANAGEMENT SYSTEM".center(70))
    print("MENU".center(70))
    print_line("*")
    print("1. Add Room")
    print("2. Delete Room")
    print("3. Display Room Details")
    print("4. Allocate Room")
    print("5. Display Room Allocation Details")
    print("6. Billing and De-Allocation")
    print("7. Save Room Allocation to a File")
    print("8. Display Room Allocation from the File")
    print("9. Backup and Clear Room Allocation File")
    print("0. Exit Application")
    print_line("*")


def run_menu():
    """Run the console-based menu until the user chooses option 0."""
    while True:
        show_menu()
        choice = input("Enter your choice number: ").strip()

        try:
            if choice == "1":
                menu_add_room()
            elif choice == "2":
                menu_delete_room()
            elif choice == "3":
                menu_display_rooms()
            elif choice == "4":
                menu_allocate_room()
            elif choice == "5":
                menu_display_allocations()
            elif choice == "6":
                menu_bill_and_deallocate()
            elif choice == "7":
                save_allocations_to_file()
            elif choice == "8":
                display_allocations_from_file()
            elif choice == "9":
                backup_and_clear_allocation_file()
            elif choice == "0":
                print("Thank you for using the Langham Hotel Management System.")
                break
            else:
                print("Invalid menu choice. Please select a number from 0 to 9.")

        except NotImplementedError as error:
            print("This part is still unfinished:", error)
        except EOFError:
            print("Input ended unexpectedly. The application will stop safely.")
            break
        except KeyboardInterrupt:
            print("\nApplication stopped by user.")
            break
        except Exception as error:
            print("Unexpected error handled safely:", error)

## 13. Task 2 - Error correction log template



In [39]:


error_correction_log = [
    {
        "number": 1,
        "error_type": "Unfinished function",
        "where_found": "delete_room()",
        "original_problem": "The function raised NotImplementedError, so the delete-room option could not work.",
        "correction_made": "Added room-number validation, existence and allocation checks, and deletion for available rooms.",
        "why_it_helped": "The program can delete an available room and prevent deletion of allocated or missing rooms."
    },
    {
        "number": 2,
        "error_type": "Unfinished function",
        "where_found": "allocate_room()",
        "original_problem": "The function was incomplete and could not create a room allocation.",
        "correction_made": "Added checks for room existence, availability, customer details, and a positive number of nights.",
        "why_it_helped": "Valid bookings can be recorded while invalid or duplicate bookings are rejected."
    },
    {
        "number": 3,
        "error_type": "Incomplete display feature",
        "where_found": "display_allocations()",
        "original_problem": "The allocation option did not display current booking details.",
        "correction_made": "Added output for room number, customer, contact, nights, room type, and nightly rate.",
        "why_it_helped": "The user can review the current booking information."
    },
    {
        "number": 4,
        "error_type": "Incomplete billing feature",
        "where_found": "bill_and_deallocate()",
        "original_problem": "The billing function did not calculate the stay total or release the room.",
        "correction_made": "Added subtotal and GST calculations, then removed the allocation and reset the room status.",
        "why_it_helped": "Billing produces a total and makes the room available again."
    },
    {
        "number": 5,
        "error_type": "Incomplete file operation",
        "where_found": "display_allocations_from_file()",
        "original_problem": "The function raised NotImplementedError instead of reading the saved report.",
        "correction_made": "Added file reading and handling for a missing file or file-access error.",
        "why_it_helped": "The saved report can be displayed, with clear messages if reading fails."
    },
    {
        "number": 6,
        "error_type": "Incomplete file operation",
        "where_found": "backup_and_clear_allocation_file()",
        "original_problem": "The function was incomplete and did not create a backup or clear the original report.",
        "correction_made": "Added a timestamped backup, then cleared the original file after the backup was written.",
        "why_it_helped": "The report is preserved in a backup while the main file is emptied."
    }
]
for item in error_correction_log:
    print(f"{item['number']}. {item['error_type']} - {item['where_found']}")

1. Unfinished function - delete_room()
2. Unfinished function - allocate_room()
3. Incomplete display feature - display_allocations()
4. Incomplete billing feature - bill_and_deallocate()
5. Incomplete file operation - display_allocations_from_file()
6. Incomplete file operation - backup_and_clear_allocation_file()


## 14. Task 2 - Required exception demonstration template



In [41]:
def demonstrate_exception_template():
    """Safe template for practising the 11 required exception types."""
    print_heading("EXCEPTION HANDLING PRACTICE")

    # 1. SyntaxError
    try:
        compile("if True print('missing colon')", "syntax_example", "exec")
    except SyntaxError as error:
        print("1. SyntaxError handled:", error.__class__.__name__)

    # 2. ValueError
    try:
        int("not-a-number")
    except ValueError as error:
        print("2. ValueError handled:", error.__class__.__name__)

    # 3. ZeroDivisionError
    try:
        result = 10 / 0
    except ZeroDivisionError as error:
        print("3. ZeroDivisionError handled:", error.__class__.__name__)

    # 4. IndexError
    try:
        sample_rooms = ["101", "102"]
        sample_rooms[5]
    except IndexError as error:
        print("4. IndexError handled:", error.__class__.__name__)

    # 5. NameError
    try:
        missing_variable_name  # noqa: F821
    except NameError as error:
        print("5. NameError handled:", error.__class__.__name__)

    # 6. TypeError
    try:
        "Room " + 101
    except TypeError as error:
        print("6. TypeError handled:", error.__class__.__name__)

    # 7. OverflowError
    try:
        very_large_value = float("1e309")
        # Python may create infinity instead of raising OverflowError,
        # so this explicit raise helps demonstrate the handler safely.
        if very_large_value == float("inf"):
            raise OverflowError("Number is too large for normal billing calculation.")
    except OverflowError as error:
        print("7. OverflowError handled:", error.__class__.__name__)

    # 8. IOError
    try:
        with open("/folder_that_does_not_exist/LHMS.txt", "r", encoding="utf-8") as file:
            file.read()
    except IOError as error:
        print("8. FileNotFoundError handled:", error.__class__.__name__)

    # 9. ImportError
    try:
        import module_that_is_not_installed_for_demo
    except ImportError as error:
        print("9. ImportError handled:", error.__class__.__name__)

    # 10. EOFError
    try:
        raise EOFError("End of input reached during menu testing.")
    except EOFError as error:
        print("10. EOFError handled:", error.__class__.__name__)

    # 11. FileNotFoundError
    try:
        with open("missing_allocation_file.txt", "r", encoding="utf-8") as file:
            file.read()
    except FileNotFoundError as error:
        print("11. FileNotFoundError handled:", error.__class__.__name__)



demonstrate_exception_template()

                     EXCEPTION HANDLING PRACTICE                      
1. SyntaxError handled: SyntaxError
2. ValueError handled: ValueError
3. ZeroDivisionError handled: ZeroDivisionError
4. IndexError handled: IndexError
5. NameError handled: NameError
6. TypeError handled: TypeError
7. OverflowError handled: OverflowError
8. FileNotFoundError handled: FileNotFoundError
9. ImportError handled: ModuleNotFoundError
10. EOFError handled: EOFError
11. FileNotFoundError handled: FileNotFoundError


## 15. Pseudocode starter for the documentation report



START Langham Hotel Management System

    SET student ID
    SET main allocation file name
    SET GST rate to 15%
    CREATE empty rooms dictionary
    CREATE empty allocations dictionary

    DEFINE helper functions to:
        display headings and divider lines
        validate room numbers
        validate positive numbers of nights
        collect valid keyboard input

    DEFINE add room:
        GET and validate room number, room type, nightly rate, and features
        IF room number already exists:
            REPORT duplicate room
        ELSE:
            ADD room with Available status
            REPORT success

    DEFINE delete room:
        GET and validate room number
        IF room does not exist:
            REPORT that it was not found
        ELSE IF room is allocated:
            REPORT that it cannot be deleted
        ELSE:
            DELETE room
            REPORT success

    DEFINE display rooms:
        IF there are no rooms:
            REPORT that no room records are available
        ELSE:
            DISPLAY each room's number, type, rate, features, and status

    DEFINE allocate room:
        GET and validate room number, customer name, contact number, and nights
        IF room does not exist or is already allocated:
            REPORT the problem
        ELSE:
            SAVE booking details in allocations
            SET room status to Allocated
            REPORT success

    DEFINE display allocations:
        IF there are no allocations:
            REPORT that no allocation details are available
        ELSE:
            DISPLAY each booking and its room details

    DEFINE bill and deallocate:
        GET and validate room number
        IF no booking exists for that room:
            REPORT that there is no allocation to bill
        ELSE:
            CALCULATE subtotal from nightly rate and nights
            CALCULATE GST at 15%
            CALCULATE total
            DISPLAY bill
            REMOVE booking from allocations
            SET room status to Available

    DEFINE build allocation report:
        CREATE report text from current allocation details
        RETURN report

    DEFINE save allocations to file:
        BUILD allocation report
        WRITE report to the main file
        REPORT success or file error

    DEFINE display allocations from file:
        READ and display the main file
        REPORT if the file is missing, empty, or cannot be read

    DEFINE backup and clear allocation file:
        CHECK that the main file exists and contains data
        CREATE a timestamped backup file
        COPY the report into the backup
        CLEAR the main file
        REPORT the result or file error

    DEFINE menu option functions:
        COLLECT required keyboard input
        CALL the matching room, allocation, billing, or file function

    REPEAT:
        DISPLAY menu options 1 to 9 and option 0 to exit
        GET the user's menu choice
        CALL the function matching the selected option
        IF the choice is invalid:
            REPORT an invalid choice
    UNTIL the user chooses 0

    DISPLAY exit message

END Langham Hotel Management System

In [42]:
run_menu()


**********************************************************************
                   LANGHAM HOTEL MANAGEMENT SYSTEM                    
                                 MENU                                 
**********************************************************************
1. Add Room
2. Delete Room
3. Display Room Details
4. Allocate Room
5. Display Room Allocation Details
6. Billing and De-Allocation
7. Save Room Allocation to a File
8. Display Room Allocation from the File
9. Backup and Clear Room Allocation File
0. Exit Application
**********************************************************************
Enter your choice number: 1
                               ADD ROOM                               
Enter room number: 111
Enter room type: King
Enter nightly rate: 525.5
Enter room features: very good
Room 111 added successfully.

**********************************************************************
                   LANGHAM HOTEL MANAGEMENT SYSTEM                    
   

In [43]:
run_menu()


**********************************************************************
                   LANGHAM HOTEL MANAGEMENT SYSTEM                    
                                 MENU                                 
**********************************************************************
1. Add Room
2. Delete Room
3. Display Room Details
4. Allocate Room
5. Display Room Allocation Details
6. Billing and De-Allocation
7. Save Room Allocation to a File
8. Display Room Allocation from the File
9. Backup and Clear Room Allocation File
0. Exit Application
**********************************************************************
Enter your choice number: 6
                      BILLING AND DE-ALLOCATION                       
Enter room number for billing: 501
Room Number : 501
Customer    : Alex Morgan
Nights      : 2
Subtotal    : $640.00
GST (15%)   : $96.00
Total       : $736.00
Billing completed. Room is now Available.

**********************************************************************
   